# Chapter 3 assignment: architectures and exact enforcement of conditions

Companion notebook for Chapter 3 of *Physics-Informed Neural Networks for Ordinary, Integral and
Integro-Differential Equations*. It runs on a CPU in a few minutes (Task 4 takes the longest).

**Learning goals**

1. Implement the Chebyshev feature map and its derivatives with the differentiated three-term
   recurrence, and verify them against automatic differentiation and the endpoint formulas.
2. Build trial functions that satisfy initial, Robin and periodic conditions *exactly* and
   verify the conditions numerically for an untrained network.
3. Count parameters of the networks of the chapter.
4. Compare an MLP and a Chebyshev network (ChNN) on the Airy oscillator
   $y'' + t\,y = 0$, $y(0)=\mathrm{Ai}(0)$, $y(10)=\mathrm{Ai}(-10)$.

Cells marked `# TODO` are for you; cells titled *Check your answer* must run without an
`AssertionError` once your solution is correct. Run the notebook from the `notebooks/` folder of
the repository.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent          # repository root (notebook lives in notebooks/)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "chapters" / "ch03_architectures"))

import numpy as np
import torch
from numpy.polynomial import chebyshev as C
from pinnode import (MLP, ChebyshevNet, ConstrainedModel, FourierFeatureNet, count_parameters,
                     derivative, rel_l2, to_tensor, train)
from pinnode.networks import chebyshev_features

torch.set_default_dtype(torch.float64)   # double precision, as everywhere in the book
torch.set_num_threads(1)
torch.manual_seed(1234)

## Recap

On $[a,b]$ the network sees the scaled input $\xi(t) = (2t-a-b)/(b-a)\in[-1,1]$.
A Chebyshev network expands $\xi$ into $T_0(\xi),\dots,T_m(\xi)$ with

$$T_0 = 1,\quad T_1=\xi,\quad T_{k+1} = 2\xi T_k - T_{k-1},$$

and differentiating the recurrence $r$ times gives, for all $\xi\in[-1,1]$ (including the endpoints),

$$T^{(r)}_{k+1} = 2\xi\,T^{(r)}_{k} + 2r\,T^{(r-1)}_{k} - T^{(r)}_{k-1},\qquad T^{(r)}_k\equiv 0\ (k<r).$$

By the chain rule, $\frac{d^r}{dt^r}T_k(\xi(t)) = \bigl(\tfrac{2}{b-a}\bigr)^r T_k^{(r)}(\xi(t))$, and
at the endpoint $T_k'(1)=k^2$, $T_k''(1) = k^2(k^2-1)/3$.

A *constrained expression* for linear conditions $\ell_j(y)=\kappa_j$ is
$y_\theta = N_\theta + \sum_j \eta_j(t)\,(\kappa_j - \ell_j(N_\theta))$ with switching functions
satisfying $\ell_i(\eta_j)=\delta_{ij}$.

## Task 1: Chebyshev features and their derivatives

Implement `cheb_features_and_derivatives(xi, m, r)` returning a list `[F0, F1, ..., Fr]` where
`Fj` is an array of shape `(len(xi), m+1)` with entries $T_k^{(j)}(\xi_i)$. Use only the
differentiated recurrence above (no `numpy.polynomial`, no closed trigonometric formulas).

In [ ]:
def cheb_features_and_derivatives(xi, m, r):
    xi = np.asarray(xi, dtype=float)
    F = [np.zeros((xi.size, m + 1)) for _ in range(r + 1)]
    # TODO: fill F[j][:, k] = T_k^{(j)}(xi) for j = 0..r and k = 0..m using the recurrences
    raise NotImplementedError
    return F

**Check your answer.**

In [ ]:
xi = np.linspace(-1.0, 1.0, 201)
m, r = 12, 2
F = cheb_features_and_derivatives(xi, m, r)
x = to_tensor(xi, requires_grad=True)
feats = chebyshev_features(x, m)
for j in range(r + 1):
    ad = np.column_stack([derivative(feats[:, k:k+1], x, j).detach().numpy().ravel() for k in range(m + 1)])
    assert np.allclose(F[j], ad, rtol=1e-10, atol=1e-10), f"derivative order {j} does not match AD"
k = np.arange(m + 1)
assert np.allclose(F[1][-1], k**2) and np.allclose(F[2][-1], k**2 * (k**2 - 1) / 3)
print("Task 1 passed: recurrences agree with automatic differentiation and the endpoint formulas.")

## Task 2: exact trial functions

For each case below, write a `transform` or a small `torch.nn.Module` such that the conditions
hold for **every** network. The check cells evaluate the conditions for an *untrained* network,
so they pass only if the conditions are built in.

(a) Second-order IVP on $[0, T]$, $T=10$: $y(0)=y_0$, $y'(0)=v_0$ with the *normalized*
multiplier $\bigl(t/T\bigr)^2$ (Proposition on trial functions for initial conditions).

(b) Dirichlet-Robin conditions on $[0,1]$: $y(0)=1$, $y'(1)+2y(1)=0$ (Exercise "A Robin trial
function").

(c) Periodic conditions on $[0, 2]$: $y(0)=y(2)$, $y'(0)=y'(2)$ using the constrained
expression with $s_1=(t-a)/L$, $s_2=(t-a)^2/L^2$.

In [ ]:
T, y0, v0 = 10.0, 0.355028053887817, 0.258819403792807

def ivp_transform_normalized(t, n):
    # TODO: return y0 + v0 t + (t/T)^2 n
    raise NotImplementedError

class RobinModel(torch.nn.Module):
    # y(0) = 1, y'(1) + 2 y(1) = 0 on [0, 1]
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, t):
        # TODO: constrained expression N + eta1 (1 - N(0)) + eta2 (0 - 2 N(1) - N'(1))
        raise NotImplementedError

class PeriodicModel(torch.nn.Module):
    # y(a) = y(b), y'(a) = y'(b) on [a, b] = [0, 2]
    def __init__(self, net, a=0.0, b=2.0):
        super().__init__()
        self.net, self.a, self.b = net, a, b
    def forward(self, t):
        # TODO: constrained expression with eta1 = (t-a)/L, eta2 = (t-a)(t-b)/(2L)
        raise NotImplementedError

**Check your answer.**

In [ ]:
def value_and_derivative(model, t0):
    t = torch.full((1, 1), t0, requires_grad=True)
    y = model(t)
    return y.item(), derivative(y, t, 1).item()

torch.manual_seed(0)
m_ivp = ConstrainedModel(MLP(1, 1, width=16, depth=2, domain=(0.0, T)), ivp_transform_normalized)
y, dy = value_and_derivative(m_ivp, 0.0)
assert abs(y - y0) < 1e-12 and abs(dy - v0) < 1e-12, "IVP conditions violated"

m_rob = RobinModel(MLP(1, 1, width=16, depth=2, domain=(0.0, 1.0)))
ya, _ = value_and_derivative(m_rob, 0.0)
yb, dyb = value_and_derivative(m_rob, 1.0)
assert abs(ya - 1.0) < 1e-12 and abs(dyb + 2 * yb) < 1e-12, "Robin conditions violated"

m_per = PeriodicModel(MLP(1, 1, width=16, depth=2, domain=(0.0, 2.0)))
ya, dya = value_and_derivative(m_per, 0.0)
yb, dyb = value_and_derivative(m_per, 2.0)
assert abs(ya - yb) < 1e-12 and abs(dya - dyb) < 1e-12, "periodic conditions violated"
print("Task 2 passed: all conditions hold to rounding error for an untrained network.")

(d) *Periodic encoding.* Build `FourierFeatureNet(frequencies=[0.5, 1.0, 1.5, 2.0], domain=(0.0, 2.0))`
and verify numerically that the network and its first two derivatives take the same values at
$t=0$ and $t=2$. Then repeat with the integer frequencies `[1, 2, 3, 4]` and show that in
addition $N(0)=N(1)$. Explain the result (Exercise "Periodicity of Fourier features").

In [ ]:
# TODO: periodic Fourier features (no check cell; compare the printed values)

## Task 3: parameter counts

The parameter count of an MLP with $L_h$ hidden layers of width $n$, input dimension $d$ and one
output is $(L_h-1)n^2 + (d+L_h+1)n + 1$. Fill in the function and compare with `count_parameters`.

In [ ]:
def n_params(L_h, n, d):
    # TODO
    raise NotImplementedError

**Check your answer.**

In [ ]:
assert n_params(2, 32, 1) == count_parameters(MLP(1, 1, width=32, depth=2)) == 1153
assert n_params(2, 32, 11) == count_parameters(ChebyshevNet(degree=10, hidden=(32, 32))) == 1473
assert n_params(2, 32, 32) == count_parameters(FourierFeatureNet(1, 1, n_features=16, width=32, depth=2))
print("Task 3 passed.")

## Task 4: MLP versus ChNN on the Airy oscillator

Train both networks with the same hard Dirichlet trial function
$y_\theta = g(t) + (1-\xi^2)N_\theta(t)$ ($g$ linear), 200 equispaced collocation points and
3000 Adam iterations followed by at most 300 L-BFGS iterations. Report the relative $L^2$ error
on 1001 points. Each training run takes about a minute on one CPU core.

Questions: Which network reaches the smaller error? Plot both predictions together with
$\mathrm{Ai}(-t)$ and describe where the MLP fails. How do you explain the difference
(think of spectral bias and of the frequency content of the features $T_0,\dots,T_{10}$)?

In [ ]:
from _ch03_common import T_END, collocation, dirichlet_normalized, residual, test_grid

def train_airy(net, n_adam=3000, n_lbfgs=300):
    model = ConstrainedModel(net, dirichlet_normalized())
    t_col = to_tensor(collocation(), requires_grad=True)
    def loss_fn():
        r = residual(model, t_col)
        L = (r ** 2).mean()
        return L, {"residual": L}
    train(model.parameters(), loss_fn, adam_iters=n_adam, lr=1e-3, lr_final=1e-5,
          lbfgs_iters=n_lbfgs, verbose=False)
    t, y_ref = test_grid()
    with torch.no_grad():
        y = model(to_tensor(t)).numpy().ravel()
    return t, y, y_ref

# TODO: train MLP(1, 1, width=32, depth=2, domain=(0, T_END)) and
#       ChebyshevNet(degree=10, hidden=(32, 32), domain=(0, T_END)) (call torch.manual_seed(1234)
#       before building each network), print rel_l2 for both and plot the predictions.

**Check your answer.** With the settings above the ChNN should reach a relative $L^2$ error well
below $10^{-2}$, while the MLP should still be far from the solution: it stays on a plateau
where it represents only the slowly varying part of $\mathrm{Ai}(-t)$. In the architecture
comparison of the chapter (4000 Adam and up to 1000 L-BFGS iterations) the MLP leaves this
plateau only during the longer L-BFGS phase. Rerun `train_airy` for the MLP with
`n_adam=4000, n_lbfgs=1000` and compare. Results can differ slightly between PyTorch versions
and machines.